# Experiment 1 — Loan Prediction (Classification)

**Course**   : ICS1512 — Machine Learning Laboratory, Semester 5  
**Roll No**  : 3122247001061  
**Dataset**  : Loan Prediction Dataset (Analytics Vidhya)  
**Aim**      : Build and evaluate classification models to predict whether a loan application will be Approved (Y) or Rejected (N).

---

**Pipeline**
```
Load Dataset → EDA → Preprocessing → Feature Selection
→ Train-Test Split → Model Training → Evaluation → Save Outputs
```

## Cell 1 — Setup: Imports & Configuration

In [ ]:
import sys
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Add ../src/ to path so we can import our modules
SRC_PATH = os.path.abspath(os.path.join("..", "src"))
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from eda              import run_eda
from preprocessing    import preprocess
from feature_selection import select_features
from models           import classification_model
from evaluation       import classification_metrics

# ---------------------------------------------------------------------------
# CONFIGURATION — change these values to adapt to a different experiment
# ---------------------------------------------------------------------------
DATASET_PATH     = os.path.join("..", "dataset", "loan_prediction", "train.csv")
TARGET_COLUMN    = "Loan_Status"       # column the model will predict
EXCLUDE_COLUMNS  = ["Loan_ID"]         # ID columns — keep in data but don't use as features
FIGURES_PATH     = os.path.abspath(os.path.join("..", "figures"))
OUTPUT_PATH      = os.path.abspath(os.path.join("..", "output"))

# Preprocessing choices
MISSING_STRATEGY = "mean"              # options: "mean", "median", "mode", "drop"
ENCODING         = "label"             # options: "label", "onehot", None
SCALING          = None                # options: "standard", "minmax", None
                                       # None is fine for Decision Tree / Naive Bayes

# Feature selection choices
FS_METHOD        = "chi2"              # options: "chi2", "anova", "mutual_info"
FS_K             = 8                   # number of top features to keep

# Model choice
MODEL_NAME       = "decision_tree"     # options: knn, decision_tree, naive_bayes,
                                       #          logistic_regression, svm, random_forest

# Train-test split
TEST_SIZE        = 0.20                # 80% train, 20% test
RANDOM_STATE     = 42                  # fixed seed for reproducibility

print("Configuration ready.")

## Cell 2 — Load Dataset

In [ ]:
df = pd.read_csv(DATASET_PATH)

print(f"Shape  : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Target : '{TARGET_COLUMN}'  →  {df[TARGET_COLUMN].unique().tolist()}")

df.head()

## Cell 3 — Exploratory Data Analysis (EDA)

In [ ]:
eda_results = run_eda(
    df            = df,
    target_column = TARGET_COLUMN,
    figures_path  = FIGURES_PATH,
)

## Cell 4 — Preprocessing

In [ ]:
clean_df = preprocess(
    df               = df,
    target_column    = TARGET_COLUMN,
    exclude_columns  = EXCLUDE_COLUMNS,
    missing_strategy = MISSING_STRATEGY,
    encoding         = ENCODING,
    scaling          = SCALING,
)

clean_df.head()

## Cell 5 — Encode Target Column

`preprocess()` deliberately protects the target column from encoding.
Feature selection functions (chi2, etc.) require a **numeric** target, so we encode it here manually.

In [ ]:
le = LabelEncoder()
clean_df[TARGET_COLUMN] = le.fit_transform(clean_df[TARGET_COLUMN])

# Print the encoding so it can be recorded in observations
mapping = dict(zip(le.classes_, le.transform(le.classes_).tolist()))
print(f"Target encoding: {mapping}")
# e.g. {'N': 0, 'Y': 1}  →  N = loan rejected, Y = loan approved

## Cell 6 — Feature Selection

In [ ]:
selected_features, scores_df = select_features(
    df              = clean_df,
    target_column   = TARGET_COLUMN,
    method          = FS_METHOD,
    k               = FS_K,
    exclude_columns = EXCLUDE_COLUMNS,
)

# Build X (features) and y (target) from the selected columns
X = clean_df[selected_features]
y = clean_df[TARGET_COLUMN]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

## Cell 7 — Train-Test Split

Kept **visible in the notebook** by design — this is a core ML step the professor checks.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y,            # keep class proportions in both splits
)

print(f"Training set : {X_train.shape[0]} samples")
print(f"Test set     : {X_test.shape[0]} samples")

## Cell 8 — Model Training

In [ ]:
model, y_pred, y_prob, train_time, pred_time = classification_model(
    X_train    = X_train,
    y_train    = y_train,
    X_test     = X_test,
    model_name = MODEL_NAME,
)

## Cell 9 — Evaluation

In [ ]:
# class_names maps integer labels back to human-readable names
# Order must match the sorted label encoding: [label_for_0, label_for_1]
# e.g. {'N': 0, 'Y': 1}  →  class_names = ["Rejected (N)", "Approved (Y)"]

metrics_df, cm_df, report = classification_metrics(
    y_test       = y_test,
    y_pred       = y_pred,
    y_prob       = y_prob,
    class_names  = ["Rejected (N)", "Approved (Y)"],
    figures_path = FIGURES_PATH,
)

print(report)

## Cell 10 — Save Outputs

In [ ]:
import os

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Save the evaluation metrics table
metrics_path = os.path.join(OUTPUT_PATH, "metrics.csv")
metrics_df.to_csv(metrics_path, index=False)
print(f"Saved: {metrics_path}")

# Save the confusion matrix
cm_path = os.path.join(OUTPUT_PATH, "confusion_matrix.csv")
cm_df.to_csv(cm_path)
print(f"Saved: {cm_path}")

# Save the feature importance ranking
scores_path = os.path.join(OUTPUT_PATH, "feature_scores.csv")
scores_df.to_csv(scores_path, index=False)
print(f"Saved: {scores_path}")

print("\nAll outputs saved.")

## Student Observations

*Record your observations after running all cells.*

| # | Observation |
|---|-------------|
| 1 | **Dataset**: ___ rows, ___ columns. Target: Loan_Status (Y/N). |
| 2 | **Missing Values**: Columns with missing data — ___. Strategy used: `mean` for numerical, `mode` for categorical. |
| 3 | **Class Imbalance**: Approved (Y): ___%, Rejected (N): ___%. |
| 4 | **Top Features** (Chi-Square): ___ |
| 5 | **Model**: Decision Tree. Accuracy: ___. F1 Score: ___. |
| 6 | **Conclusion**: ___ |